# Automatic Deep Research - Adding custom tools

Welcome to the second practice lab of this module! 

In this lab, you will continue working on the deep research crew from Lesson 1. This time you will be writing you own custom tools, and adding them to your agents so that they can give more accurate responses.

**What you'll learn:**
- How to create custom tools for your agents

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an AI research crew that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter.

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Problem statement](#1)
- [2. Set up your notebook](#2)
- [3. Tools](#3)
- [4. Agents](#4)
- [5. Guardrails](#5)
- [6. Tasks](#6)
- [7. Execution hooks](#7)
- [8. Crew](#8)
  - [8.1. Define the crew](#8-1)
  - [8.2. Define the inputs](#8-2)
  - [8.3. Run the crew](#8-3)

<a id="1"></a>

## 1. Problem statement

The goal of this lab is to take the multi-agent system that can interpret a user's input, and create an action plan, then do the actual research and fact checking, and finally output a report you can share with the client, and add tools to the agents so they can be better at achieving their goals.

You will reuse the code from the first practice lab of this module, so you only need to write new code in the sections [Tools](#3), [Agents](#4), and [Define the inputs](#8.2), the rest of the lab remains the same, with the solution to the previous lab already given to you.

Here is a visual summary of the structure of your crew, as well as the new elements you will be adding: 

<img src="../images/lab2-agents-tasks-diagram.png">


<a id="2"></a>

## 2. Set up your notebook

Begin by setting up the notebook by importing all necessary modules, and configuring the environment variables so you can connect to OpenAI.

In [1]:
# Patch to disable SSL verification for Coursera
from patch import disable_ssl_verification
disable_ssl_verification()

from crewai import Agent, Task, Crew, LLM
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
import os
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key, get_exa_api_key
from IPython.display import Markdown
import yaml

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key 
os.environ["OPENAI_API_KEY"] = get_openai_api_key()
# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

<a id="3"></a>

## 3. Tools

The final goal of this Crew you've been building during the course is to provide the user with a complete report containing researched information about a topic, and what is a report without some cool graphics?

In the next cell, you will be writing a custom tool that automatically creates charts based on a report of gathered information. This tool will be added to the **Report Writer** agent, so it can add visualization into the final report. 

Remember that the base structure for a custom tool is
```python
class MyCustomTool(BaseTool):
    name: str = "Name of my tool"
    description: str = "What this tool does. It's vital for effective utilization."
    args_schema: Type[BaseModel] = {}

    def _run(self, argument: str) -> str:
        # Your tool's logic here
        return "Tool's result"
```

In this case, you will need to complete the `CustomPlotTool` class with: 
- `name`: a suitable name for the tool
- `description`: This should be a detailed description of the tool. Mention:
    - The expected input: the full validated information, as a string
    - What it does: automatically generates plots from text
- `_run()` function: specify the type of input and output expected by the tool

The code for generating the plots is already given to you.

In [3]:
# import packages needed for the custom tool
from crewai.tools import BaseTool
from crewai import LLM
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json

# Define the custom tool for creating plots
class CustomPlotTool(BaseTool):
    ### START CODE HERE ###
    name: str = "Final report visualization tool"
    description: str = "This is a custom tool that automatically creates charts based on the final report of gathered information {research}. As input the gather information {research} is expected as string."
    def _run(self, research: str) -> str:
    ### END CODE HERE ###
        try:
            extraction_prompt = f"""
            You are an expert data visualization assistant. Analyze the provided research text and identify meaningful, insightful charts that can be created to visualize quantifiable data supporting the research's key insights and findings. Only suggest charts for data that includes numerical values, measurable trends, comparisons, or categorical distributions that can be effectively plotted.

            Focus on creating visualizations that highlight trends, comparisons, distributions, or relationships that add value to the research. Avoid suggesting charts for purely qualitative or non-quantifiable information.

            For each chart, provide a JSON object with:
              - "chart_type" (string: choose from "line" for trends over time/continuous, "bar" for comparisons, "histogram" for distributions, "scatter" for relationships, "pie" for proportions)
              - "x_axis" (string: variable name for x-axis, e.g., "year", "category")
              - "y_axis" (string: variable name for y-axis, e.g., "value", "count")
              - "color" (string: optional variable for color grouping/hue, or null if not applicable)
              - "Title" (string: descriptive, insightful title that explains what the chart shows)
              - "data" (dictionary: keys matching x_axis, y_axis, and color variables; values as lists of extracted numerical/categorical data from the research)

            Ensure data is accurately extracted and formatted as lists. If a variable has multiple series (e.g., for color), include all in the data dictionary.

            If no quantifiable data suitable for meaningful visualization is present in the research, return an empty array [].

            Text:
            {research}

            Example output (return valid JSON only):
            [
              {{"chart_type": "line", "x_axis": "year", "y_axis": "funding_amount", "color": "sector", "Title": "AI Research Funding Trends by Sector", "data": {{"year": [2020, 2021, 2022], "funding_amount": [2.5, 3.8, 5.2], "sector": ["Healthcare", "Finance", "Tech"]}}}},
              {{"chart_type": "bar", "x_axis": "tool_name", "y_axis": "adoption_rate", "color": null, "Title": "Market Adoption Rates of AI Tools", "data": {{"tool_name": ["ToolA", "ToolB", "ToolC"], "adoption_rate": [45, 67, 23]}}}}
            ]

            Return only the JSON array, no additional text or explanations.
            """
            llm = LLM(model="gpt-4o-mini",)  # Initialize the LLM instance
            llm_response = llm.call([{"role": "user", "content": extraction_prompt}])

            # Clean the response to extract just the JSON part
            llm_response = llm_response.strip()
            if llm_response.startswith('```json'):
                llm_response = llm_response[7:]  # Remove ```json
            if llm_response.endswith('```'):
                llm_response = llm_response[:-3]  # Remove ```
            llm_response = llm_response.strip()

            # --- Step 2: Parse the LLM output ---
            charts_data = json.loads(llm_response)

            if not isinstance(charts_data, list) or len(charts_data) == 0:
                return "No information found in the research to visualize."

            plots_created = []

            # --- Step 3: Create plots for each chart ---
            for i, chart_info in enumerate(charts_data):
                try:
                    # Extract chart configuration
                    chart_type = chart_info.get("chart_type", None).lower()
                    x_axis = chart_info.get("x_axis", "x")
                    y_axis = chart_info.get("y_axis", "y") 
                    title = chart_info.get("Title", f"Chart {i+1}")
                    hue = chart_info.get("color", None)
                    data = chart_info.get("data", {})

                    # Create DataFrame from the data
                    df = pd.DataFrame(data)

                    if df.empty:
                        continue

                    # Create the plot
                    plt.figure(figsize=(10, 6))

                    if chart_type == "line":
                        sns.lineplot(data=df, x=x_axis, y=y_axis, marker="o", hue=hue)
                    elif chart_type in ["bar", "column"]:
                        sns.barplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "histogram":
                        plt.hist(df[y_axis], bins=10, alpha=0.7, hue=hue)
                        plt.xlabel(y_axis)
                        plt.ylabel("Frequency")
                    elif chart_type == "scatter":
                        # Default to scatter plot
                        sns.scatterplot(data=df, x=x_axis, y=y_axis, hue=hue)
                    elif chart_type == "pie":
                        # For pie chart, assume y_axis is values, x_axis is labels
                        plt.pie(df[y_axis], labels=df[x_axis], autopct='%1.1f%%', startangle=90)
                        plt.title(title)
                        plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.

                    plt.title(title)
                    plt.xticks(rotation=45)
                    plt.tight_layout()

                    # --- Step 4: Save the plot ---
                    os.makedirs("plots", exist_ok=True)
                    filename = f"plots/plot_{i+1}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
                    plt.savefig(filename, dpi=300, bbox_inches='tight')
                    plt.close()

                    plots_created.append(filename)

                except Exception as e:
                    print(f"Error creating chart {i+1}: {str(e)}")
                    continue

            if plots_created:
                return f"Successfully created {len(plots_created)} plots: {', '.join(plots_created)}"
            else:
                return "No plots could be created from the extracted data."

        except json.JSONDecodeError as e:
            return f"Error parsing LLM response as JSON: {str(e)}"
        except Exception as e:
            return f"Error generating smart plot: {str(e)}"

As in the previous labs, you will still use the web search and scraping tools for the **Internet Researcher** and **Fact Checker** agents, which you will initialize in the next cell

In [4]:
# create tools instances
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
scrape_website_tool = ScrapeWebsiteTool()

<a id="4"></a>

## 4. Agents

For this system, you will use four agents:
- Research Planner
- Internet Researcher
- Fact checker
- Report Writer

All their arguments (`role`, `goal`, `backstory`) are already given to you in a YAML file, which you will use to configure the agents. If you want to take a closer look go to the [config/agents.yaml](config/agents.yaml) file on the file navigator on the left.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

Don't forget to add the new custom tool to the **Report Writer** agent!

In [7]:
# load the configuration file for the agents
with open('config/agents.yaml', 'r') as file:
        agent_config = yaml.safe_load(file)


# create the agents using the configuration
research_planner = Agent(
        config=agent_config['research_planner'],
        verbose=True,
        max_rpm=150,
        max_iter=15
        )

internet_researcher = Agent( 
        config=agent_config['internet_researcher'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

fact_checker = Agent(
        config=agent_config['fact_checker'],
        verbose=True,
        tools=[exa_search_tool, scrape_website_tool],
        max_rpm=150,
        max_iter=15
        )

report_writer = Agent(
        config=agent_config['report_writer'],
        verbose=True,
        ### START CODE HERE ### 
        # add the automatic plot tool
        tools=[CustomPlotTool()],
        ### END CODE HERE ###
        max_rpm=150,
        max_iter=15
        )

<a id="5"></a>

## 5. Guardrails

To make your system more robust, you want to add guardrails to your tasks. These guardrails provide a way to validate and transform task outputs before they are passed to the next task, helping ensure data quality and providing feedback to agents when their output doesn't meet specific criteria. You can find out more about guardrails in the [docs](https://docs.crewai.com/en/concepts/tasks#task-guardrails).


In particular, you will implement a guardrail for the final output. You want to make sure the final report has all the sections needed: 
- Summary
- Insights (or recommendations)
- Citations (or References)

In [8]:
import re

# write the custom guardrail function
def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw 
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )
    
    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False, 
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False, 
                "The report must include an Insights section with a header like '## Insights'"
                )

    # check that the citations (or references) section exists
    if not re.search(r'#+.*citations|#+.*references', output_lower): 
        return (False, 
                "The report must include a Citations (or References) section with a header like '## Citations'"
                )
    return (True, output)

<a id="6"></a>

## 6. Tasks
Now you are ready to create the tasks. Just as you did with the agents, you will load the configuration from a YAML file, which you can find in [`config/tasks.yaml`](config/tasks.yaml).  

In order to actually add the charts to the final report you will need to update the `write_final_report` task in the [`tasks.yaml`](config/tasks.yaml) file. Adapt the `description` and `expected_output` to include instructions include the charts generated by the custom tool.

Once that's done, run the next cell to create each task. The agents, guardrails, and context are already defined.

In [9]:
# load the configuration file for the tasks
with open('config/tasks.yaml', 'r') as file:
    task_config = yaml.safe_load(file)


# create the tasks using the configuration
create_research_plan = Task( 
    config=task_config['create_research_plan'],
    agent=research_planner 
)

gather_research_data = Task(
    config=task_config['gather_research_data'],
    agent=internet_researcher,
)

verify_information_quality = Task(
    config=task_config['verify_information_quality'],
    agent=fact_checker, 
)

write_final_report = Task( 
    config=task_config['write_final_report'],
    agent=report_writer, 
    guardrails=[write_report_guardrail],
)

<a id="7"></a>

## 7. Execution hooks

The last step before creating the Crew is creating an [after kickoff hook](https://docs.crewai.com/en/learn/before-and-after-kickoff-hooks#after-kickoff-hook). 

In this case, you will create a hook that takes the final output and saves it to a Markdown file on your local file system. 

In [10]:
def save_file_hook(result):
    """
    Save the final research report to a local markdown file
    """
    try:
        # Get the final report content from the last task output
        if hasattr(result, 'tasks_output') and result.tasks_output:
            report_content = result.tasks_output[-1].raw
        else:
            report_content = str(result)
        
        filename = f"research_report-p2.md"
        
        # Save to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
        
        print(f"Report successfully saved to: {filename}")
        
    except Exception as e:
        print(f"Error saving report to file: {str(e)}")

<a id="8"></a>

## 8. Crew

<a id="8-1"></a>

### 8.1. Define the crew
Run the next cell to define the crew. 

In [11]:
# Create the urban planning crew
deep_research_crew = Crew(
    # include all the agents
    agents=[research_planner, 
            internet_researcher, 
            fact_checker, 
            report_writer],
    # include all the tasks in the order to be executed
    tasks=[create_research_plan, 
           gather_research_data, 
           verify_information_quality, 
           write_final_report],
    # add memory to the crew
    memory=True,
    # add the after kickoff hook
    after_kickoff_callbacks=[save_file_hook]
)

<a id="8-2"></a>

### 8.2. Define the inputs

Use the next cell to define the inputs to your Crew. This should represent the user's query. Try using the same query as in the previous lab, so you can compare the results.

In [12]:
### START CODE HERE ###

# write your query in the "user_query" value
inputs = { 
        "user_query": "Create a summary of US author's Edgar Allan Poe's literary work as a poet."
}
### END CODE HERE ###   

<a id="8-3"></a>

### 8.3. Run the crew
Now you can run, or kick off, the crew to get the result.

In [13]:
# Execute the crew's tasks
result = deep_research_crew.kickoff(inputs=inputs)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Task: Break down the research query "Create a summary of US author's Edgar Allan Poe's literary work as a      │
│  poet." into specific topics and key questions that need investigation. Create a focused research plan.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Planner                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan to Analyze "Create a summary of US author's Edgar Allan Poe's literary work as a poet."**      │
│                                                                                                                 │
│  1) **Main Research Topics to Investigate:**                                                                    │
│     - A. Overview of Edgar Allan Poe's Life and Influences                                                      │
│     - B. Major Themes and Motifs in Poe's Poetry                                                                │
│     - C. Analysis of Key Poems by Edgar Allan Poe                                                               │
│     - D. Poe's Poetic Style and Techniques                                                                      │
│     - E. Reception and Legacy of Poe's Poetry                                                                   │
│                                                                                                                 │
│  2) **Key Questions for Each Topic:**                                                                           │
│                                                                                                                 │
│     A. **Overview of Edgar Allan Poe's Life and Influences**                                                    │
│     - What are the significant events in Edgar Allan Poe's life that influenced his writing?                    │
│     - Who were the key literary influences on Poe's poetic work?                                                │
│     - How did Poe's personal struggles and experiences shape his poetry?                                        │
│                                                                                                                 │
│     B. **Major Themes and Motifs in Poe's Poetry**                                                              │
│     - What themes (e.g., death, love, madness) are prevalent in Poe's poetry?                                   │
│     - How does Poe utilize symbolism in conveying these themes?                                                 │
│     - In what ways do his themes reflect the broader Romantic movement?                                         │
│                                                                                                                 │
│     C. **Analysis of Key Poems by Edgar Allan Poe**                                                             │
│     - What are Poe's most significant poems and what makes them noteworthy?                                     │
│     - How do individual poems exemplify the themes and motifs identified?                                       │
│     - What techniques does Poe use in specific poems to enhance their impact?                                   │
│                                                                                                                 │
│     D. **Poe's Poetic Style and Techniques**                                                                    │
│     - What are the distinguishing features of Poe's poetic style?                                               │
│     - How does Poe's use of meter, rhyme, and rhythm contribute to his poetry?                                  │
│     - In what ways do figurative language and imagery p

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Using the research plan, search the internet and scrape relevant websites to collect comprehensive       │
│  information on all identified topics. Verify information across multiple sources and cite all sources used.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Thought: I need to start gathering comprehensive information on Edgar Allan Poe's life as well as his poetry.  │
│  I'll begin by searching for an overview of his life and influences, which will set the foundation for          │
│  understanding his work.                                                                                        │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "Overview of Edgar Allan Poe's Life and Influences",                                         │
│    "start_published_date": null,                                                                                │
│    "end_published_date": null,                                                                                  │
│    "include_domains": null                                                                                      │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Thought: I have effectively gathered and synthesized comprehensive information about Edgar Allan Poe's         │
│  literary work as a poet. This includes an overview of his life, major themes in his poetry, analysis of key    │
│  poems, his poetic style and techniques, and the reception and legacy of his work. I will now proceed to        │
│  create a clear and structured final report based on this information.                                          │
│                                                                                                                 │
│  Using Tool: Final report visualization tool                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "research": "Edgar Allan Poe: A Summary of His Literary Work as a Poet\n\n1. **Overview of Edgar Allan       │
│  Poe's Life and Influences**\n    - Poe was born on January 19, 1809, in Boston, Massachusetts, to actors       │
│  David and Elizabeth Poe. After the death of his mother in 1811, he was raised by John and Frances Allan in     │
│  Richmond, Virginia. Despite his talents, financial difficulties plagued him throughout his life, affecting     │
│  his education and career choices (Britannica, n.d.; Wikipedia, n.d.).\n    - His tumultuous relationship with  │
│  Allan, coupled with the tragic loss of his young wife Virginia to tuberculosis, significantly influenced his   │
│  literary voice, often reflected in themes of loss and grief in his poetry (PBS, 2006; Poetry Foundation,       │
│  n.d.).\n\n2. **Major Themes and Motifs in Poe's Poetry**\n    - **Death and Grief**: Death is a recurring      │
│  theme, explored through poems like “The Raven” and “Annabel Lee,” where the loss of a beloved is a source of   │
│  profound sorrow and existential despair (PapersOwl, 2025; LitCharts, n.d.).\n    - **Love and Loss**: Many of  │
│  Poe's poems embody a tragic connection to lost loves, often portraying the female figure as a representation   │
│  of beauty and innocence (Gale, n.d.; GradeSaver, n.d.).\n    - **Madness and the Subconscious**: Themes of     │
│  madness and psychological introspection, as illustrated in “The Tell-Tale Heart,” are prevalent in many of     │
│  his works, often blurring the lines between sanity and insanity (Aithor, 2024).\n    - **The Gothic and the    │
│  Supernatural**: Poe's poetry frequently incorporates elements of gothic horror, creating an atmosphere of      │
│  dread and exploring the supernatural in relation to human emotions (Cambridge University Press, 2008).\n\n3.   │
│  **Analysis of Key Poems**\n    - **“The Raven” (1845)**: This poem features a narrator mourning the loss of    │
│  his beloved Lenore. The refrain of \"Nevermore\" encapsulates the speaker's descent into madness and despair,  │
│  signifying an eternal separation from love (Poetry Foundation, n.d.).\n    - **“Annabel Lee” (1849)**: Often   │
│  interpreted as an homage to his own lost love, Virginia, this poem addresses themes of undying love amid       │
│  separation by death, with a haunting yet romantic tone (World Poetry Collective, 2024).\n    - **“Ulalume”     │
│  (1847)**: This poem employs rich symbolism, with the kingdom by the sea as a metaphor for idyllic love that    │
│  ultimately succumbs to darkness (eapoe.org, n.d.).\n    - **“A Dream within a Dream” (1849)**: Here, Poe       │
│  contemplates the nature of reality and the elusiveness of time, reinforcing his overarching theme of           │
│  impermanence and loss (Academia.edu, 2022).\n\n4. **Poe's Poetic Style and Techniques**\n    - **Unique        │
│  Structure and Sound**: Poe’s use of meter, rhyme, and various forms of sound devices, such as alliteration     │
│  and assonance, enhanced the musicality of his poetry, making it both impactful and memorable (American         │
│  Literature, 2025).\n    - **Imagery and Symbolism**: He employed vivid imagery and complex symbolism to evoke  │
│  deep emotional resonances, often relating to love, death, and the human condition (Poe Studies, 2024).\n\n5.   │
│  **Reception and Legacy of Poe's Poetry**\n    - *Contemporaneous Reception*: Initially, Poe's poetry received  │
│  mixed reviews, with some critics praising his originality while others deemed his work as shallow or           │
│  excessively morbid (Edgar Allan Poe Society of Baltim

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  No information found in the research to visualize.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Writer                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Edgar Allan Poe: A Summary of His Literary Work as a Poet                                                    │
│                                                                                                                 │
│  ## Executive Summary                                                                                           │
│                                                                                                                 │
│  Edgar Allan Poe, an influential figure in American literature, is renowned for his dark and macabre poetry.    │
│  His exploration of profound themes such as death, love, and madness, combined with his unique poetic           │
│  techniques, has left an indelible mark on both American and world literature. This report analyzes his life    │
│  influences, major themes and motifs in his poetry, key poems, poetic style, and the reception and legacy of    │
│  his work.                                                                                                      │
│                                                                                                                 │
│  ## Detailed Findings                                                                                           │
│                                                                                                                 │
│  ### 1. Overview of Edgar Allan Poe's Life and Influences                                                       │
│  - Edgar Allan Poe was born on January 19, 1809, in Boston, Massachusetts, to actors David and Elizabeth Poe.   │
│  The early death of his mother led to his upbringing by John and Frances Allan in Richmond, Virginia. Despite   │
│  possessing literary talent, Poe faced financial struggles throughout his life, which affected his education    │
│  and career (Britannica, n.d.; Wikipedia, n.d.).                                                                │
│  - His difficult relationship with Allan and the tragic loss of his young wife, Virginia, to tuberculosis       │
│  profoundly impacted his literary voice, often reflected in the themes of loss and grief evident in his poetry  │
│  (PBS, 2006; Poetry Foundation, n.d.).                                                                          │
│                                                                                                                 │
│  ### 2. Major Themes and Motifs in Poe's Poetry                                                                 │
│  - **Death and Grief**: Death is a recurring theme in Poe's poetry, depicted in works like "The Raven" and      │
│  "Annabel Lee," where he examines profound sorrow and existential despair stemming from the loss of loved ones  │
│  (PapersOwl, 2025; LitCharts, n.d.).                                                                            │
│  - **Love and Loss**: Many of Poe's poems embody a tragic connection to lost loves, portraying the female       │
│  figure as a symbol of beauty and innocence (Gale, n.d.; GradeSaver, n.d.).                                     │
│  - **Madness and the Subconscious**: Themes of madness and explorations of the subconscious are prevalent in    │
│  his works, often illustrating the blurred lines between sanity and insanity (Aithor, 2024).                    │
│  - **The Gothic and the Supernatural**: Poe's poetry fr

Report successfully saved to: research_report-p2.md


After it finishes running, you should be able to see the newly created Markdown file with your report in the file navigator on the left. You can compare it with the results from the first lab, can you see any differences?

Congratulations! You've successfully completed this lab 🎉